In [61]:
import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass, field

In [62]:
class ContinuousPPO(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim):
        super(ContinuousPPO, self).__init__()
        self.feature_extractor = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.CELU(),
        )
        self.actor_mean = nn.Sequential(
            nn.Linear(hidden_dim, action_dim),
        )
        self.actor_logstd = nn.Parameter(torch.zeros(1, action_dim))
        self.critic = nn.Linear(hidden_dim, 1)

    def forward(self, state):
        features = self.feature_extractor(state)
        action_mean = self.actor_mean(features)
        action_std = self.actor_logstd.expand_as(action_mean).exp()
        state_value = self.critic(features)  
        return action_mean, action_std, state_value

    def act(self, state):
        action_mean, action_std, state_value = self.forward(state)
        dist = torch.distributions.Normal(action_mean, action_std)
        action = dist.sample()
        return action, dist.log_prob(action).sum(dim=-1), state_value

In [63]:
@dataclass
class Trainer:
    env: gym.Env
    total_timesteps: int = 500000
    rollout_length: int = 2048
    n_epochs: int = 10
    batch_size: int = 64
    c1: float = 0.5  
    c2: float = 0.01  
    gae: float = 0.95
    gamma: float = 0.99
    clip_epsilon: float = 0.2

    def __post_init__(self):
        self.model = ContinuousPPO(
            state_dim=self.env.observation_space.shape[0],
            action_dim=self.env.action_space.shape[0],
            hidden_dim=64
        )

        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=3e-4)
        self.buffer = []
        self.state, _ = self.env.reset()

    def _rollout(self, num_steps):
        self.buffer = []
        for _ in range(num_steps):
            state_tensor = torch.FloatTensor(self.state).unsqueeze(0)
            with torch.no_grad():
                action_tensor, log_prob_tensor, value_tensor = self.model.act(state_tensor)
            action_np = action_tensor.cpu().numpy()[0]
            # Pendulum action range is [-2, 2], Tanh output is [-1, 1]
            action_clipped = np.clip(action_np * 2.0, -2.0, 2.0)
            next_state, reward, terminated, truncated, _ = self.env.step(action_clipped)
            done = terminated or truncated
            self.buffer.append((self.state, action_np, reward, log_prob_tensor.item(), value_tensor.item(), float(done)))
            self.state = next_state
            if done: 
                self.state, _ = self.env.reset()  
                     
        next_state_tensor = torch.FloatTensor(self.state).unsqueeze(0)
        with torch.no_grad():
            _, _, next_value_tensor = self.model.act(next_state_tensor)
            next_value = 0.0 if done else next_value_tensor.item()
        return next_value

    def _compute_gae(self, next_value):
        advantages = []
        gae = 0
        for step in reversed(self.buffer):
            _, _, reward, _, value, done = step
            # delta = r + gamma * V(s') * (1-done) - V(s)
            delta = reward + self.gamma * next_value * (1 - done) - value
            gae = delta + self.gamma * self.gae * (1 - done) * gae
            advantages.insert(0, gae)
            next_value = value
        return torch.FloatTensor(advantages)

    def train(self):
        timestep = 0
        self.state, _ = self.env.reset()
        while timestep < self.total_timesteps:
            next_value = self._rollout(self.rollout_length)
            timestep += self.rollout_length
            advs = self._compute_gae(next_value)
            # Normalize advantages
            advs = (advs - advs.mean()) / (advs.std() + 1e-8)
            # Prepare batches
            s_b = torch.FloatTensor(np.array([x[0] for x in self.buffer]))
            a_b = torch.FloatTensor(np.array([x[1] for x in self.buffer]))
            lp_old_b = torch.FloatTensor([x[3] for x in self.buffer])
            v_old_b = torch.FloatTensor([x[4] for x in self.buffer])
            ret_b = advs + v_old_b
            for _ in range(self.n_epochs):
                indices = np.random.permutation(self.rollout_length)
                for start in range(0, self.rollout_length, self.batch_size):
                    idx = indices[start : start + self.batch_size]
                    # Forward pass
                    mu, std, val = self.model(s_b[idx])
                    dist = torch.distributions.Normal(mu, std)
                    lp_new = dist.log_prob(a_b[idx]).sum(dim=-1)
                    ent = dist.entropy().mean()
                    # Actor Loss (PPO Clip)
                    ratio = (lp_new - lp_old_b[idx]).exp()
                    surr1 = ratio * advs[idx]
                    surr2 = torch.clamp(ratio, 1 - self.clip_epsilon, 1 + self.clip_epsilon) * advs[idx]
                    actor_loss = -torch.min(surr1, surr2).mean()
                    # Critic Loss
                    critic_loss = F.mse_loss(val.squeeze(-1), ret_b[idx])
                    # Total Loss
                    loss = actor_loss + self.c1 * critic_loss - self.c2 * ent
                    self.optimizer.zero_grad()
                    loss.backward()
                    nn.utils.clip_grad_norm_(self.model.parameters(), 0.5)
                    self.optimizer.step()
            if timestep % 10 * self.rollout_length == 0:
                print(f"Step: {timestep} | Loss: {loss.item():.4f}")

In [64]:
env = gym.make('Pendulum-v1')
trainer = Trainer(env=env, total_timesteps=300000)
trainer.train()

Step: 10240 | Loss: 0.1562
Step: 20480 | Loss: 0.4217
Step: 30720 | Loss: 0.1504
Step: 40960 | Loss: 0.1562
Step: 51200 | Loss: 0.2493
Step: 61440 | Loss: 0.3700
Step: 71680 | Loss: 0.3545
Step: 81920 | Loss: 0.1861
Step: 92160 | Loss: 0.1707
Step: 102400 | Loss: 0.4494
Step: 112640 | Loss: 0.5248
Step: 122880 | Loss: 0.4019
Step: 133120 | Loss: 0.3670
Step: 143360 | Loss: 0.1750
Step: 153600 | Loss: 0.1700
Step: 163840 | Loss: 0.4246
Step: 174080 | Loss: 0.2580
Step: 184320 | Loss: 0.1436
Step: 194560 | Loss: 0.2080
Step: 204800 | Loss: 0.2664
Step: 215040 | Loss: 0.4471
Step: 225280 | Loss: 0.2285
Step: 235520 | Loss: 0.4857
Step: 245760 | Loss: 0.0877
Step: 256000 | Loss: 0.5007
Step: 266240 | Loss: 0.0896
Step: 276480 | Loss: 0.1686
Step: 286720 | Loss: 0.0574
Step: 296960 | Loss: 0.1853


In [65]:
env_test = gym.make('Pendulum-v1', render_mode='human')
env_test.metadata['render_fps'] = 120
seed = 123
state, _ = env_test.reset(seed=seed)

episode_reward = 0
episode_count = 1
for step in range(1, 601):
    state_t = torch.FloatTensor(state).unsqueeze(0)
    with torch.no_grad():
        mu, _, _ = trainer.model(state_t)
    action = mu.cpu().numpy()[0] * 2.0
    state, reward, done, trunc, _ = env_test.step(action)
    episode_reward += reward
    if done or trunc:
        print(f"Episode {episode_count} kết thúc | Reward: {episode_reward:.2f}")
        state, _ = env_test.reset()
        episode_reward = 0
        episode_count += 1
env_test.close()

Episode 1 kết thúc | Reward: -131.73
Episode 2 kết thúc | Reward: -133.35
Episode 3 kết thúc | Reward: -275.41
